# ch05 练习：重复值与异常值

> 共 8 道基础题 + 1 道综合题（拆 2 步），合计 **10 个空**
> 建议限时：**30 分钟**

数据：`data/device_defects.csv`（270 行 x 13 列）

**这章练的是「口径」**。同一个问题换一个口径，答案可能差一倍甚至十倍。
每题都给了验收断言，跑通即正确。

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.simplefilter("ignore")
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

plt.rcParams["font.sans-serif"] = ["PingFang SC", "Heiti SC", "Arial Unicode MS", "STHeiti"]
plt.rcParams["axes.unicode_minus"] = False

dev = pd.read_csv("data/device_defects.csv")
BIZ_KEY = ["台区编号", "线路名称", "设备类型", "设备编号", "缺陷类型", "缺陷等级", "发现日期"]
SENTINEL = [9999.0, -1.0, 5000.0]
load = dev["负荷值"]

print("读入 device_defects.csv ->", dev.shape)

---

### TODO(1) 完全重复：报数 + 去重

1. 完全重复行数存入 `n_dup_full`
2. 去重后的 DataFrame 存入 `d_dedup`（并重置索引）

**考点**：默认 `keep="first"`，此时 `sum()` 就是**应删的行数**。

In [ ]:
# TODO(1)：完全重复行数 -> n_dup_full；drop_duplicates 并重置索引 -> d_dedup
# 提示：DataFrame.duplicated().sum() ；DataFrame.drop_duplicates(ignore_index=True)
n_dup_full = ____
d_dedup = ____

# ---- 验收 ----
assert n_dup_full == 6, f"完全重复行应为 6，实际 {n_dup_full}"
assert d_dedup.shape == (264, 13), f"去重后应为 (264, 13)，实际 {d_dedup.shape}"
assert d_dedup.duplicated().sum() == 0, "去重后不应还有完全重复行"
assert list(d_dedup.index) == list(range(len(d_dedup))), "别忘了 ignore_index=True 重置索引"
print(f"TODO(1) 通过 -> 完全重复 {n_dup_full} 行，去重后 {d_dedup.shape}")

### TODO(2) 业务键重复：报数 + 去重

1. 按 `BIZ_KEY`（业务键，已在上方定义）判重复，行数存入 `n_dup_biz`
2. 按业务键去重后的 DataFrame 存入 `d_dedup_biz`（重置索引）

**注意**：这个口径比完全重复**多**，因为 `记录ID` / `备注` 不同但业务内容相同的行也被算进来。

In [ ]:
# TODO(2)：业务键重复行数 -> n_dup_biz；按业务键去重 -> d_dedup_biz
# 提示：DataFrame.duplicated(subset=列名列表).sum() ；DataFrame.drop_duplicates(subset=..., ignore_index=True)
n_dup_biz = ____
d_dedup_biz = ____

# ---- 验收 ----
assert n_dup_biz == 10, f"业务键重复行应为 10，实际 {n_dup_biz}"
assert d_dedup_biz.shape == (260, 13), f"业务键去重后应为 (260, 13)，实际 {d_dedup_biz.shape}"
assert n_dup_biz > n_dup_full, "业务键口径应该比完全重复口径多"
assert d_dedup_biz.duplicated(subset=BIZ_KEY).sum() == 0, "业务键去重后不应还有业务重复"
print(f"TODO(2) 通过 -> 业务键重复 {n_dup_biz} 行（比完全重复多 {n_dup_biz - n_dup_full} 行）")

### TODO(3) `keep` 语义：为什么数字会翻倍

1. `keep=False` 口径下完全重复的掩码检出数存入 `n_keep_false`
2. `drop_duplicates(keep=False)` 后的形状行数存入 `n_rows_keep_false`

**考点**：`keep=False` 会把重复组**所有成员**都标 True，
所以 `n_keep_false` 是 `n_dup_full` 的 2 倍，**不能当"要删的行数"**。

In [ ]:
# TODO(3)：keep=False 的检出数 -> n_keep_false；keep=False 去重后的行数 -> n_rows_keep_false
# 提示：DataFrame.duplicated(keep=False).sum() ；DataFrame.drop_duplicates(keep=False)
n_keep_false = ____
n_rows_keep_false = ____

# ---- 验收 ----
assert n_keep_false == 12, f"keep=False 应检出 12 行，实际 {n_keep_false}"
assert n_keep_false == n_dup_full * 2, "keep=False 的结果正好是 keep='first' 的 2 倍"
assert n_rows_keep_false == 258, f"keep=False 去重后应为 258 行，实际 {n_rows_keep_false}"
assert n_rows_keep_false < 264, "keep=False 会把重复组两端都删掉，比 keep='first' 删得更多"
print(f"TODO(3) 通过 -> keep=False 检出 {n_keep_false} 行（= 2 x {n_dup_full}），去重后剩 {n_rows_keep_false} 行")

### TODO(4) 物理边界法：湿度的越界值

1. 湿度列中 `> 100` 或 `< 0` 的个数存入 `n_humid_bad`
2. 按物理边界 `[0, 100]` 截断后的湿度存入 `humid_clipped`
3. 截断后仍越界的个数存入 `n_humid_bad_after`

**考点**：有**明确物理边界**的字段永远优先用业务规则，不要用统计法。

In [ ]:
humid = dev["湿度"]

# TODO(4)：越界个数 -> n_humid_bad；clip(0,100) -> humid_clipped；截断后越界数 -> n_humid_bad_after
# 提示：布尔条件用 | 连接并各自加括号；Series.clip(lower, upper)
n_humid_bad = ____
humid_clipped = ____
n_humid_bad_after = ____

# ---- 验收 ----
assert n_humid_bad == 5, f"湿度越界应为 5 个，实际 {n_humid_bad}"
assert n_humid_bad_after == 0, f"截断后不应还有越界，实际 {n_humid_bad_after}"
assert humid_clipped.max() == 100 and humid_clipped.min() >= 0, "截断后的范围应是 [0, 100]"
assert humid_clipped.isna().sum() == 0, "湿度列本来没有缺失，截断不应制造缺失"
print(f"TODO(4) 通过 -> 越界 {n_humid_bad} 个，截断后 {n_humid_bad_after} 个")

### TODO(5) 全局 IQR：算出围栏并数异常

`负荷值` 里掺了 `9999` / `-1` / `5000` 三种哨兵码。先看全局 IQR 会判出多少个：

1. 下四分位存入 `q1`，上四分位存入 `q3`，四分位距存入 `iqr`
2. IQR 围栏下界存入 `iqr_lo`，上界存入 `iqr_hi`
3. 落在围栏外的个数存入 `n_out_iqr`

**提醒**：这个数会**远大于**真实哨兵码个数（11 个）——因为负荷值是多峰的。

In [ ]:
# TODO(5)：q1 / q3 / iqr / 围栏上下界 / 围栏外个数
# 提示：Series.quantile([0.25, 0.75]) 一次拿两个；围栏 = q1-1.5*iqr, q3+1.5*iqr；越界计数用 | 连接两个条件
q1, q3 = ____
iqr = ____
iqr_lo = ____
iqr_hi = ____
n_out_iqr = ____

# ---- 验收 ----
assert abs(q1 - 48.31) < 1e-6, f"q1 应为 48.31，实际 {q1}"
assert abs(q3 - 192.34) < 1e-6, f"q3 应为 192.34，实际 {q3}"
assert abs(iqr - 144.03) < 1e-6, f"IQR 应为 144.03，实际 {iqr}"
assert abs(iqr_lo - -167.735) < 1e-6, f"下界应为 -167.735，实际 {iqr_lo}"
assert abs(iqr_hi - 408.385) < 1e-6, f"上界应为 408.385，实际 {iqr_hi}"
assert n_out_iqr == 52, f"全局 IQR 应判出 52 个，实际 {n_out_iqr}"
assert n_out_iqr > 11, "这个数远大于真哨兵码 11 个 -> 全局 IQR 在多峰数据上误杀严重"
print(f"TODO(5) 通过 -> 围栏 [{iqr_lo:.3f}, {iqr_hi:.3f}]，判出 {n_out_iqr} 个（真哨兵码只有 11 个）")

### TODO(6) 全局 3σ：被哨兵码污染的典型

1. 用 `mean ± 3*std` 算出上下界，存入 `sig_lo`、`sig_hi`
2. 落在区间外的个数存入 `n_out_3sigma`

**看完数字请回答**：区间宽度是多少？正常负荷的取值范围是多少？
这个区间还有筛选力吗？为什么 `-1` 一个都没被抓到？

In [ ]:
# TODO(6)：3σ 上下界 -> sig_lo, sig_hi；区间外个数 -> n_out_3sigma
# 提示：Series.mean() / Series.std() ；界 = mean - 3*std, mean + 3*std
mu = ____
sd = ____
sig_lo = ____
sig_hi = ____
n_out_3sigma = ____

# ---- 验收 ----
assert abs(mu - 379.68457831325304) < 1e-6, f"均值应为 379.6846，实际 {mu}"
assert abs(sd - 1351.7630979420865) < 1e-6, f"标准差应为 1351.7631，实际 {sd}"
assert abs(sig_lo - -3675.6047) < 0.01, f"下界应约 -3675.60，实际 {sig_lo:.4f}"
assert abs(sig_hi - 4434.9739) < 0.01, f"上界应约 4434.97，实际 {sig_hi:.4f}"
assert n_out_3sigma == 7, f"3σ 应判出 7 个，实际 {n_out_3sigma}"
assert (sig_hi - sig_lo) > 8000, "区间宽度 8000+，而正常负荷只有 5~590 -> 完全没有筛选力"
assert int((load < 0).sum()) == 4, "数据里有 4 个 -1，但 3σ 下界是 -3675 -> 一个都没抓到"
print(f"TODO(6) 通过 -> 3σ 区间 [{sig_lo:.2f}, {sig_hi:.2f}]，判出 {n_out_3sigma} 个，漏掉全部 4 个 -1")

### TODO(7) 分组 IQR：多峰分布的正解

按 `设备类型` 分组，对每组的 `负荷值` 各自做 IQR 判定：

1. 布尔掩码（与原始行一一对应）存入 `grp_out_mask`
2. 掩码为 True 的个数存入 `n_out_grp`

**考点**：`groupby(...).transform(函数)` 把"每组算一个界"的结果**广播回每一行**，
索引天然对齐，可直接当掩码用。**这是处理多峰数据的标准姿势。**

In [ ]:
def iqr_mask(s: pd.Series, k: float = 1.5) -> pd.Series:
    """对一组做 IQR 围栏判定，返回布尔掩码。"""
    q1g, q3g = s.quantile([0.25, 0.75])
    iqrg = q3g - q1g
    return (s < q1g - k * iqrg) | (s > q3g + k * iqrg)


# TODO(7)：按设备类型分组 IQR 掩码 -> grp_out_mask；异常个数 -> n_out_grp
# 提示：DataFrame.groupby(键)[列].transform(函数) ；结果再 .astype(bool)
grp_out_mask = ____
n_out_grp = ____

# ---- 验收 ----
assert isinstance(grp_out_mask, pd.Series), "grp_out_mask 应该是 Series"
assert len(grp_out_mask) == len(dev), "掩码长度应等于行数（transform 会广播回每行）"
assert set(grp_out_mask.unique()) <= {True, False}, "掩码只能是布尔值"
assert n_out_grp == 10, f"分组 IQR 应判出 10 个，实际 {n_out_grp}"
assert n_out_grp < n_out_iqr, "分组 IQR 远比全局 IQR 精确"
assert int((grp_out_mask & (load >= 1000)).sum()) == 7, "7 个 >=1000 的哨兵码应全部命中"
assert int((grp_out_mask & (load < 0)).sum()) == 2, "只能命中 4 个 -1 中的 2 个"
print(f"TODO(7) 通过 -> 分组 IQR 判出 {n_out_grp} 个（全局 IQR 是 {n_out_iqr} 个，误杀 {n_out_iqr - n_out_grp} 个）")
print("  仍漏掉 2 个 -1 -> 说明统计法必须配业务规则兜底")

### TODO(8) 两阶段清洗：哨兵 → NaN → 填充 → clip

这是本章的核心流程，**顺序不能反**：

1. `s1`：把 `SENTINEL`（`9999` / `-1` / `5000`）替换成 `np.nan`
2. `s2`：用 `s1` 的中位数填充（按 ch04 的策略）
3. 用 `s2` 重算 IQR 围栏（`lo2` / `hi2`），判出仍偏高的个数存入 `n_out_2`
4. `clean_load`：把 `s2` 截断到 `[lo2, hi2]`

**如果先 clip 再判哨兵，假值会被剪成"看起来正常"的数**——这是最常见的错误。

In [ ]:
# TODO(8)：哨兵置 NaN -> s1；中位数填充 -> s2；重算围栏 -> lo2/hi2；越界数 -> n_out_2；截断 -> clean_load
# 提示：Series.replace({旧值: np.nan, ...}) ；Series.fillna(Series.median()) ；再用 quantile 重算围栏
s1 = ____
s2 = ____
q1b, q3b = ____
iqrb = ____
lo2 = ____
hi2 = ____
n_out_2 = ____
clean_load = ____

# ---- 验收 ----
assert int(s1.isna().sum()) == 32, f"哨兵置 NaN 后应有 32 个缺失（原 21 + 哨兵 11），实际 {int(s1.isna().sum())}"
assert int(s2.isna().sum()) == 0, "填充后不应还有缺失"
assert abs(s2.median() - 86.05) < 1e-6, f"填充后中位数应为 86.05，实际 {s2.median()}"
assert abs(s2.mean() - 156.678) < 1e-3, f"填充后均值应约 156.678，实际 {s2.mean():.4f}"
assert abs(lo2 - -119.9212) < 1e-3, f"新下界应约 -119.9212，实际 {lo2:.4f}"
assert abs(hi2 - 348.5687) < 1e-3, f"新上界应约 348.5687，实际 {hi2:.4f}"
assert n_out_2 == 45, f"新围栏下应判出 45 个，实际 {n_out_2}"
assert abs(clean_load.mean() - 128.246) < 1e-3, f"截断后均值应约 128.246，实际 {clean_load.mean():.4f}"
assert int(clean_load.isna().sum()) == 0, "清洗后不应有缺失"
assert clean_load.max() <= hi2 + 1e-9, "截断后最大值不应超过上界"
print(f"TODO(8) 通过 -> 哨兵 11 个置 NaN，填充后 mean {s2.mean():.3f} -> 截断后 {clean_load.mean():.3f}")
print(f"  std: {load.std():.2f}（原始，被污染）-> {clean_load.std():.2f}（清洗后）")

---

## 综合题：从清洗到可视化

| 步骤 | 产物 | 对应评分点 |
|---|---|---|
| ① 分组箱线图 | `boxes` | 数据可视化 · 箱线图（2 分） |
| ② 组装报告 | `report` | 数据探索 · 结论输出 |

#### 综合 ① 按设备类型的箱线图

画出按 `设备类型` 分组的 `负荷值` 箱线图，共 **5 个箱体**。

1. `fig, ax` 用 `plt.subplots`
2. `boxes` 接住 `ax.boxplot(...)` 的返回值（后面要用它数箱体个数）
3. 给图加标题（中文别忘设置字体，前面已经设过）

**提示**：`boxplot` 需要"每组一个数组"，用
`[g.dropna().to_numpy() for _, g in dev.groupby("设备类型")["负荷值"]]`。

In [ ]:
# TODO(9)：画按设备类型分组的负荷值箱线图：fig/ax 用 plt.subplots，返回值存入 boxes
# 提示：ax.boxplot(数组列表, tick_labels=标签列表) ；数组列表用列表推导式从 groupby 取每组数组
fig, ax = ____
boxes = ____

ax.set_title("按设备类型的负荷值箱线图（多峰 + 哨兵码）")
ax.set_ylabel("负荷值")
plt.tight_layout()
plt.show()

# ---- 验收 ----
assert len(boxes["boxes"]) == 5, f"应有 5 个箱体，实际 {len(boxes['boxes'])}"
assert len(boxes["fliers"]) == 5, "每组都应画出离群点（个数可以为 0）"
flier_max = max(
    (float(f.get_ydata().max()) if len(f.get_ydata()) else 0.0)
    for f in boxes["fliers"]
)
assert flier_max > 1000, f"应能看到 9999 这个离群点，实际最大离群值 {flier_max}"
print("综合 ① 通过 -> 已画出 5 组箱线图，各组离群点数：",
      [len(f.get_ydata()) for f in boxes["fliers"]])
print("  读图结论：箱体高度差异极大（变压器 ~521 vs 绝缘子 ~55）-> 分布是多峰的")
print("            -> 全局统计口径必然误判，必须分组定界")

#### 综合 ② 组装清洗报告

把这一章的结论组装成 `report` 字典，键必须完全一致：

```
n_rows / n_dup_full / n_dup_biz / n_dup_after_dedup
n_sentinel / n_out_iqr / n_out_grp / n_humid_bad
clean_mean / clean_max
```

其中：
- `n_dup_after_dedup`：按业务键去重后**仍有的完全重复行数**（应为 0）
- `n_sentinel`：哨兵码总数（`>= 1000` 的与 `< 0` 的之和）
- `clean_mean` / `clean_max`：两阶段清洗后 `clean_load` 的均值与最大值（保留 4 位小数取整）

In [ ]:
# TODO(10)：组装 report 字典（10 个键）
# 提示：键名必须完全一致；n_sentinel = (load>=1000).sum() + (load<0).sum()；clean_mean 用 round(float(...), 4)
report = ____

# ---- 验收 ----
expect_keys = {
    "n_rows", "n_dup_full", "n_dup_biz", "n_dup_after_dedup", "n_sentinel",
    "n_out_iqr", "n_out_grp", "n_humid_bad", "clean_mean", "clean_max",
}
assert isinstance(report, dict), "report 应该是字典"
assert set(report) == expect_keys, (
    f"键不对。多的是 {set(report) - expect_keys}，缺的是 {expect_keys - set(report)}"
)
assert report["n_rows"] == 270
assert report["n_dup_full"] == 6 and report["n_dup_biz"] == 10
assert report["n_dup_after_dedup"] == 0, "业务键去重后不应还有完全重复行"
assert report["n_sentinel"] == 11, "哨兵码应为 11 个（7 个 >=1000 + 4 个负值）"
assert report["n_out_iqr"] == 52 and report["n_out_grp"] == 10
assert report["n_humid_bad"] == 5
assert abs(report["clean_mean"] - 128.246) < 1e-3
assert abs(report["clean_max"] - 348.5687) < 1e-3
assert report["n_out_grp"] < report["n_out_iqr"], "分组口径应比全局口径精确"
print("综合 ② 通过 -> 清洗报告：")
for k, v in report.items():
    print(f"  {k:22s} = {v!r}")

---

### 复盘提问

**复盘问自己**（四题都能答上来才算过）：

1. `duplicated(keep=False).sum()` 得到 12，题目问"有多少重复行"，能填 12 吗？为什么？
2. 为什么 `9999` 不能 `clip` 到 408？正确做法是什么？
3. 全局 IQR 判出 52 个、分组 IQR 判出 10 个。哪个对？错的那个错在哪？
4. 湿度的 5 个越界值，为什么用业务规则比用 IQR 更靠谱？

答不上来的，回 `_notes/错题本.md` 记一笔。